# Aufbereitung Bevölkerungsdaten

Quelle: BFS, *Ständige und nichtständige Wohnbevölkerung nach Jahr, Kanton, Bevölkerungstyp, Staatsangehörigkeit, Geburtsstaat und Geschlecht* (px-x-0103010000_499), 2010 bis 2025.

**Warum nicht `pyaxis` wie bei den Kriminalitätsdaten?** Die Datei enthält 109 Mio. Zellen (jede Staatsangehörigkeit × jeder Geburtsstaat). `pyaxis` baut daraus ein riesiges DataFrame und braucht über 6 GB RAM. Darum lesen wir die Zahlen direkt mit numpy ein (ca. 1.2 GB, 30 Sekunden).

**Was behalten wird:** alles, also alle Jahre, Kantone, Bevölkerungstypen, Staatsangehörigkeiten, Geburtsstaaten und Geschlechter inklusive aller Kombinationen. Weggelassen werden nur Zellen mit dem Wert 0 (97.6 % der Datei). Dabei geht keine Information verloren: Eine Kombination, die fehlt, hat 0 Personen. Fehlende Werte gibt es in der Datei keine.

**Wichtig beim Arbeiten mit den Daten:** Bei `pivot_table` immer `fill_value=0` setzen, damit fehlende Kombinationen als 0 statt als NaN erscheinen.

Gespeichert wird als **Parquet** (kleiner und schneller als CSV).

In [ ]:
import re
import numpy as np
import pandas as pd

## 1. PX-Datei einlesen

In [ ]:
text = open("Bevolkerungsdaten.px", encoding="utf-8-sig").read()
kopf, daten = text.split("DATA=", 1)
del text

# Dimensionen (Reihenfolge der Daten) und ihre Ausprägungen aus dem Kopf lesen
liste = lambda s: re.findall(r'"([^"]*)"', s)
dims = liste(re.search(r"STUB=(.*?);", kopf, re.S).group(1)) + liste(re.search(r"HEADING=(.*?);", kopf, re.S).group(1))
werte = {d: liste(re.search(r'VALUES\("%s"\)=(.*?);' % re.escape(d), kopf, re.S).group(1)) for d in dims}
{d: len(v) for d, v in werte.items()}

In [ ]:
# Zahlen direkt als numpy-Array einlesen (fehlende Werte wie "..." würden zu NaN)
daten = re.sub(r'"[^"]*"', " nan ", daten).rstrip().rstrip(";")
arr = np.fromstring(daten, sep=" ")
del daten

arr = arr.reshape([len(werte[d]) for d in dims])
print(arr.shape, "Zellen:", arr.size, "| fehlende Werte:", np.isnan(arr).sum(), f"| Nullen: {(arr == 0).mean():.1%}")

## 2. Kategorie „Ausland“ ergänzen

Bei Staatsangehörigkeit und Geburtsstaat wird je eine Kategorie `"Ausland"` hinzugefügt (Total minus Schweiz). Weil das für alle Kombinationen gemacht wird, gibt es z. B. auch:

- Staatsangehörigkeit Ausland + Geburtsstaat Schweiz: Ausländer, die in der Schweiz geboren sind (zweite Generation)
- Staatsangehörigkeit Schweiz + Geburtsstaat Ausland: eingebürgerte bzw. im Ausland geborene Schweizer
- Staatsangehörigkeit Ausland + Geburtsstaat Ausland: zugewanderte Ausländer

In [ ]:
for d in ["Staatsangehörigkeit", "Geburtsstaat"]:
    achse = dims.index(d)
    total = arr.take([werte[d].index(f"{d} - Total")], axis=achse)
    schweiz = arr.take([werte[d].index("Schweiz")], axis=achse)
    arr = np.concatenate([arr, total - schweiz], axis=achse)
    werte[d] = werte[d] + ["Ausland"]

print(arr.shape, "| kleinster Wert (darf nicht negativ sein):", arr.min())

## 3. In eine Tabelle umwandeln (ohne Nullen)

In [ ]:
nz = np.nonzero(arr)  # Positionen aller Zellen ungleich 0, je ein Index-Array pro Dimension

bev = pd.DataFrame({d: pd.Categorical.from_codes(nz[k], werte[d]) for k, d in enumerate(dims)})
bev["Jahr"] = np.array(werte["Jahr"], dtype=int)[nz[0]]
bev["DATA"] = arr[nz]
del arr, nz

print(bev.shape)
bev.head(10)

## 4. Plausibilitätscheck

In [ ]:
# Ständige Wohnbevölkerung Neuchâtel: Total, Schweiz, Ausland und Ausländeranteil
ne = bev[(bev["Kanton"] == "Neuchâtel") & (bev["Bevölkerungstyp"] == "Ständige Wohnbevölkerung")
         & (bev["Geschlecht"] == "Geschlecht - Total") & (bev["Geburtsstaat"] == "Geburtsstaat - Total")
         & bev["Staatsangehörigkeit"].isin(["Staatsangehörigkeit - Total", "Schweiz", "Ausland"])]
ne = ne.pivot_table(index="Jahr", columns="Staatsangehörigkeit", values="DATA", fill_value=0, observed=True)
ne["Ausländeranteil %"] = ne["Ausland"] / ne["Staatsangehörigkeit - Total"] * 100
ne.round(1)

In [ ]:
t = bev[(bev["Geschlecht"] == "Geschlecht - Total") & (bev["Geburtsstaat"] == "Geburtsstaat - Total")]

# Summe der Kantone muss dem Total Schweiz entsprechen
s = t.pivot_table(index=["Jahr", "Bevölkerungstyp", "Staatsangehörigkeit"], columns="Kanton", values="DATA", fill_value=0, observed=True)
print("Grösste Abweichung Kantone vs. Schweiz:", (s.drop(columns="Schweiz").sum(axis=1) - s["Schweiz"]).abs().max())

# Summe der einzelnen Länder muss dem Total entsprechen (ohne Total und ohne die neue Kategorie Ausland)
gruppen = ["Jahr", "Kanton", "Bevölkerungstyp"]
laender = t[~t["Staatsangehörigkeit"].isin(["Staatsangehörigkeit - Total", "Ausland"])].groupby(gruppen, observed=True)["DATA"].sum()
total = t[t["Staatsangehörigkeit"] == "Staatsangehörigkeit - Total"].groupby(gruppen, observed=True)["DATA"].sum()
print("Grösste Abweichung Länder vs. Total:", (laender - total).abs().max())

In [ ]:
# Die 10 grössten ausländischen Gruppen in Neuchâtel 2016
t[(t["Kanton"] == "Neuchâtel") & (t["Jahr"] == 2016) & (t["Bevölkerungstyp"] == "Ständige Wohnbevölkerung")
  & ~t["Staatsangehörigkeit"].isin(["Staatsangehörigkeit - Total", "Schweiz", "Ausland"])].nlargest(10, "DATA")[["Staatsangehörigkeit", "DATA"]]

## 5. Als Parquet speichern

In [ ]:
bev.to_parquet("bevoelkerung.parquet", index=False)
# Später einlesen mit: bev = pd.read_parquet("bevoelkerung.parquet")

## 6. Kriminalitätsdaten von CSV in Parquet umwandeln

`beschuldigte.csv` ist 4.6 GB gross. Die Textspalten werden beim Einlesen direkt als Kategorie gespeichert, das spart viel Arbeitsspeicher. Danach reicht in den anderen Notebooks `pd.read_parquet("beschuldigte.parquet")`.

Diese Zelle muss nur **einmal** laufen.

In [ ]:
import os
spalten = ["Straftat", "Kanton", "Aufenthaltsgruppe", "Geschlecht", "Altersklasse"]
df_b = pd.read_csv("beschuldigte.csv", dtype={c: "category" for c in spalten})
df_b.to_parquet("beschuldigte.parquet", index=False)

print(df_b.shape)
print(f"CSV:     {os.path.getsize('beschuldigte.csv') / 1e9:.2f} GB")
print(f"Parquet: {os.path.getsize('beschuldigte.parquet') / 1e9:.2f} GB")